In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType

In [0]:

df_ocorrencias_silver = spark.table("projeto_seguranca_rj.bronze.ocorrencias_municipio")

In [0]:
df_ocorrencias_silver.dropDuplicates().count()

In [0]:
total = df_ocorrencias_silver.count()

sem_duplicatas = df_ocorrencias_silver.dropDuplicates().count()

print("Total:", total)
print("Após remover duplicatas:", sem_duplicatas)
print("Duplicatas:", total - sem_duplicatas)

In [0]:

duplicados_chave = (
    df_ocorrencias_silver
    .groupBy(
        "fmun_cod",
        "ano",
        "mes"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

display(duplicados_chave)

In [0]:
display(df_ocorrencias_silver.limit(5))


In [0]:
df_ocorrencias_silver.columns

In [0]:
df_ocorrencias_silver = (
    df_ocorrencias_silver
    .withColumn(
        "data_referencia",
        F.to_date(
            F.concat_ws(
                "-",
                F.col("ano"),
                F.lpad(F.col("mes"),2,"0"),
                F.lit("01")
            )
        )
    )
)

In [0]:
df_ocorrencias_silver.select(
    [
        F.count(
            F.when(
                F.col(c).isNull(),
                c
            )
        ).alias(c)
        for c in df_ocorrencias_silver.columns
    ]
).show()

In [0]:
colunas_nulas = [
    "feminicidio",
    "tentativa_feminicidio"
]

for coluna in colunas_nulas:
    df_ocorrencias_silver = (
        df_ocorrencias_silver
        .withColumn(
            coluna,
            F.coalesce(
                F.col(coluna),
                F.lit(0)
            )
        )
    )

In [0]:
df_ocorrencias_silver = (
    df_ocorrencias_silver
    .dropna(
        subset=[
            "fmun_cod",
            "fmun",
            "ano",
            "mes"
        ]
    )
)

In [0]:

df_ocorrencias_silver = (
    df_ocorrencias_silver
    .withColumn(
        "fmun_cod",
        F.col("fmun_cod").cast(StringType())
    )
)

In [0]:
df_ocorrencias_silver.printSchema()

In [0]:
df_ocorrencias_silver.show()

In [0]:
# Salvar ocorrências

df_ocorrencias_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "projeto_seguranca_rj.silver.ocorrencias_municipio"
    )
